In [2]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch_geometric.data import Data
#from torch_geometric.nn import gcn_norm
from torch_geometric.nn.conv.gcn_conv import gcn_norm



from tqdm import tqdm

In [3]:
import pickle
with open('/home/iiitb/Desktop/anant/GridRaster/part_ours_training/new_dict.pkl', 'rb') as f:
    supp_dict = pickle.load(f)


from torch.utils.data import DataLoader
from dataset import PartQueryDataset, custom_transform

# root directory pointing to training_data
dataset_root = "/home/iiitb/Desktop/anant/GridRaster/part_ours_training/data/training_data_MOHAN"

# supp_dict is defined above

dataset = PartQueryDataset(root_dir=dataset_root, supp_dict=supp_dict, transform=custom_transform)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=8)

In [ ]:
# data = np.load("../../GridRaster/part_ours_training/saved_features/item_11.npz", allow_pickle=True)

# query_dict = data['query_dict']
# support_dict = data['support_dict']
# query_full_superpixels = data['query_full_superpixels']
# support_part_superpixels = data['support_part_superpixels']
# gt_query_part_superpixels = data['gt_query_part_superpixels']
# cos_mat_dist = data['cos_mat_dist']

#print(query_dict.shape, support_dict.shape, query_full_superpixels.shape, support_part_superpixels.shape, gt_query_part_superpixels.shape, cos_mat_dist.shape)

() () (44,) (7,) (8,) (44, 7)


In [4]:
from part_seg import get_query_feature_and_affinity_matrix
import numpy as np
import os

In [5]:
dataloader.dataset.__len__()/8

539.875

In [5]:
for batch in dataloader:
    print(len(batch)) # batch size=8
    print(batch.keys()) # 'query_image', 'query_full_mask', 'query_part_mask', 'object_id', 'part_id', 'support_image', 'support_full_mask', 'support_part_mask'
    print(batch['query_image'].shape) # batch_size*224*224
    print(len(batch['query_image']))
    print(batch['object_id'])
    print(batch['part_id'])
    break

8
dict_keys(['query_image', 'query_full_mask', 'query_part_mask', 'object_id', 'part_id', 'support_image', 'support_full_mask', 'support_part_mask'])
torch.Size([8, 224, 224])
8
tensor([11, 19, 36,  6,  6,  1,  8, 41])
tensor([ 13, 184,  70,  22,  23,  77,  46, 196])


In [10]:
from torch_geometric.data import Data, Batch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

processed_data_dir = "processed_data"
os.makedirs(processed_data_dir, exist_ok=True)

idx = 0
# for single batch_size
for batch in dataloader:
    batched_data_list = []

    # the below code is to get the item
    for i in range(len(batch['query_image'])):
        query_dict, support_dict, query_full_superpixels, support_part_superpixels, gt_query_part_superpixels, cos_mat_dist = get_query_feature_and_affinity_matrix(batch["support_image"][i], batch["support_part_mask"][i], 
                                                                                                     batch["support_full_mask"][i], 
                                                                                                     batch["query_image"][i], batch["query_part_mask"][i], 
                                                                                                     batch["query_full_mask"][i])
        
        #print(query_dict.keys(), support_dict.keys(), query_full_superpixels.shape, support_part_superpixels.shape, gt_query_part_superpixels.shape, cos_mat_dist.shape)
        #print(query_full_superpixels.shape, support_part_superpixels.shape, gt_query_part_superpixels.shape, cos_mat_dist.shape)

        # Get node features for query object
        X = torch.tensor(query_dict['superpixel_features'][query_full_superpixels], dtype=torch.float32)
        # Get node features for support part
        X_part = torch.tensor(support_dict['superpixel_features'][support_part_superpixels], dtype=torch.float32)



        # below code is to create cos_mat_dist
        adj = cos_mat_dist @ cos_mat_dist.T
        adj = torch.tensor(adj, dtype=torch.float32)

        # Build graph edge_index and edge_weight from cos_mat_dist
        edge_index = (adj > 0).nonzero(as_tuple=False).t()
        edge_weight = adj[edge_index[0], edge_index[1]]



        # Superpixel remapping
        id_map = {orig_id: new_id for new_id, orig_id in enumerate(query_full_superpixels)}
        segments = query_dict['superpixel_labels']

        # Create a mask of where superpixel labels are in query_full_superpixels
        mask = np.isin(query_dict['superpixel_labels'], query_full_superpixels)
        filtered_superpixels = np.where(mask, query_dict['superpixel_labels'], 0)
        segments = filtered_superpixels

        gt_query_part_seg = np.isin(segments, gt_query_part_superpixels)

        # Generate superpixel-level labels
        superpixel_gt_labels = np.full(len(id_map), -1)
        for orig_id, new_id in id_map.items():
            mask = segments == orig_id
            labels = gt_query_part_seg[mask]
            if len(labels) == 0:
                continue
            values, counts = np.unique(labels, return_counts=True)
            superpixel_gt_labels[new_id] = values[np.argmax(counts)]

        # Replace -1 with 0 (assumed background)
        superpixel_gt_labels[superpixel_gt_labels == -1] = 0

        # Create `y` tensor
        y = torch.tensor(superpixel_gt_labels, dtype=torch.long)

        img_t = batch['query_image'][i].cpu()



        # this below code is added to have batch infor in the pyG dataloader for the x_part also.
        # Number of nodes in x_part for this graph
        num_part_nodes = X_part.size(0)
        # For a single graph, batch_part is just all zeros
        batch_part = torch.zeros(num_part_nodes, dtype=torch.long)


        # Build PyG Data object (storing object_d and part_id which will be used in evaluation (optional for Training))
        data = Data(x=X, x_part=X_part, batch_part=batch_part, edge_index=edge_index, edge_weight=edge_weight, y=y, object_id = batch['object_id'][i], part_id = batch['part_id'][i])

        data.segments = torch.tensor(segments, dtype=torch.long)  # [H, W]
        data.query_full_superpixels = torch.tensor(query_full_superpixels, dtype=torch.long)
        data.gt_query_part_superpixels = torch.tensor(gt_query_part_superpixels, dtype=torch.long)
        data.image = img_t
        
        # Save data object
        torch.save(data, os.path.join(processed_data_dir, f"graph_{idx}.pt"))
        idx += 1

        # Append to list for batching
        #batched_data_list.append(data)

        # print(batched_data_list)
        
        break # just to print one element in the batch

    #print(batched_data_list)    
    break #to just print first batch

# Convert the list of Data objects to a single batched object
#batch_data = Batch.from_data_list(batched_data_list)

#print(batch_data)

# # Move to device
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# batch_data = batch_data.to(device)

# # Now batch_data can be passed to the model
# output = model(batch_data.x, batch_data.edge_index, batch_data.edge_weight)
# print(output)

In [11]:
data

Data(x=[134, 1024], edge_index=[2, 17956], y=[134], x_part=[15, 1024], batch_part=[15], edge_weight=[17956], object_id=33, part_id=104, segments=[224, 224], query_full_superpixels=[134], gt_query_part_superpixels=[34], image=[224, 224])

In [12]:
data.batch_part

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

## Below code is same as above Data but we are loading from the saved path

In [13]:
GRAPH_PATH = "/home/iiitb/Desktop/anant/GridRaster/train_processed_data/graph_0.pt"
data = torch.load(GRAPH_PATH)
data

Data(x=[20, 1024], edge_index=[2, 400], y=[20], x_part=[4, 1024], batch_part=[4], edge_weight=[400], object_id=9, part_id=60, segments=[224, 224], query_full_superpixels=[20], gt_query_part_superpixels=[2], image=[224, 224])

In [14]:
data_list = []
for i in range(8):
    GRAPH_PATH = f"/home/iiitb/Desktop/anant/GridRaster/train_processed_data/graph_{i}.pt"
    data = torch.load(GRAPH_PATH)
    data_list.append(data)
data_list

[Data(x=[20, 1024], edge_index=[2, 400], y=[20], x_part=[4, 1024], batch_part=[4], edge_weight=[400], object_id=9, part_id=60, segments=[224, 224], query_full_superpixels=[20], gt_query_part_superpixels=[2], image=[224, 224]),
 Data(x=[41, 1024], edge_index=[2, 1681], y=[41], x_part=[6, 1024], batch_part=[6], edge_weight=[1681], object_id=21, part_id=37, segments=[224, 224], query_full_superpixels=[41], gt_query_part_superpixels=[12], image=[224, 224]),
 Data(x=[19, 1024], edge_index=[2, 361], y=[19], x_part=[4, 1024], batch_part=[4], edge_weight=[361], object_id=13, part_id=110, segments=[224, 224], query_full_superpixels=[19], gt_query_part_superpixels=[5], image=[224, 224]),
 Data(x=[139, 1024], edge_index=[2, 19321], y=[139], x_part=[12, 1024], batch_part=[12], edge_weight=[19321], object_id=2, part_id=68, segments=[224, 224], query_full_superpixels=[139], gt_query_part_superpixels=[10], image=[224, 224]),
 Data(x=[9, 1024], edge_index=[2, 81], y=[9], x_part=[3, 1024], batch_part=[

In [3]:
20+41+19+139+9+22+28+15

293

In [15]:
from graph_dataset import GraphPartDataset
from torch_geometric.loader import DataLoader as GeoDataLoader

BATCH_SIZE = 8

dataset = GraphPartDataset("/home/iiitb/Desktop/anant/GridRaster/train_processed_data")  # load .pt files
loader = GeoDataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

for batch in loader:
    print(batch)
    #print(batch.batch)
    #print(batch.y)
    break

DataBatch(x=[610, 1024], edge_index=[2, 74152], y=[610], x_part=[93, 1024], batch_part=[93], edge_weight=[74152], object_id=[8], part_id=[8], segments=[1792, 224], query_full_superpixels=[610], gt_query_part_superpixels=[133], image=[1792, 224], batch=[610], ptr=[9])


In [16]:
for batch in loader:
    print("batch.x.shape:", batch.x.shape)  # e.g. [483, 1024]
    print("batch.ptr:", batch.ptr)          # length = num_graphs + 1
    # per-graph node counts:
    per_graph_counts = torch.diff(batch.ptr)    # requires PyTorch >=1.8
    # or alternative:
    # per_graph_counts = torch.bincount(batch.batch)
    print("per-graph node counts:", per_graph_counts.tolist())
    break


batch.x.shape: torch.Size([610, 1024])
batch.ptr: tensor([  0,  20,  61, 210, 254, 453, 512, 566, 610])
per-graph node counts: [20, 41, 149, 44, 199, 59, 54, 44]


In [17]:
import torch, os

root_dir = "/home/iiitb/Desktop/anant/GridRaster/train_processed_data"
files = sorted(os.listdir(root_dir))[:8]  # string sort
# OR numeric sort:
# files = sorted(os.listdir(root_dir), key=lambda x: int(x.split('_')[1].split('.')[0]))[:8]

for f in files:
    data = torch.load(os.path.join(root_dir, f))
    print(f, data.x.size(0), data.x_part.size(0), data.edge_index.shape, getattr(data, 'object_id', None), getattr(data, 'part_id', None))


graph_0.pt 20 4 torch.Size([2, 400]) tensor(9) tensor(60)
graph_1.pt 41 6 torch.Size([2, 1681]) tensor(21) tensor(37)
graph_10.pt 149 8 torch.Size([2, 22201]) tensor(27) tensor(0)
graph_11.pt 44 7 torch.Size([2, 1936]) tensor(27) tensor(5)
graph_12.pt 199 17 torch.Size([2, 39601]) tensor(31) tensor(87)
graph_13.pt 59 8 torch.Size([2, 3481]) tensor(6) tensor(24)
graph_14.pt 54 25 torch.Size([2, 2916]) tensor(12) tensor(98)
graph_15.pt 44 18 torch.Size([2, 1936]) tensor(35) tensor(91)


In [11]:
4+6+8+46+18+9+14+18

123

In [9]:
data.x.shape, data.x_part.shape

(torch.Size([90, 1024]), torch.Size([18, 1024]))

In [ ]:
type(data.x)